In [2]:
print("Hello, World!")

Hello, World!


In [3]:
%pwd

'c:\\Users\\susov\\OneDrive\\Desktop\\MoodSphere\\AI\\research'

In [4]:
import os
print(os.listdir())

['analysis.ipynb', 'trials.ipynb']


In [5]:
os.chdir("../")
%pwd

'c:\\Users\\susov\\OneDrive\\Desktop\\MoodSphere\\AI'

In [6]:
# importing necessary libraries
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\susov\OneDrive\Desktop\MoodSphere\AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# function to load pdf files from a directory
def load_pdf(file_path):
    load_pdf=DirectoryLoader(
        file_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    return load_pdf.load()


In [8]:
extracted_data=load_pdf("data")
extracted_data

[]

In [9]:
len(extracted_data)

0

In [10]:
from typing import List
from langchain.schema import Document

# function to filter documents to only include page content and source metadata
def filter_to_minimal_docs(docs:List[Document])->List[Document]:
    minimal_docs=[]
    for doc in docs:
        minimal_doc=Document(
            page_content=doc.page_content,
            metadata={"source":doc.metadata["source"]}
        )
        minimal_docs.append(minimal_doc)
    return minimal_docs

In [11]:
minimal_docs=filter_to_minimal_docs(extracted_data)
minimal_docs

[]

In [12]:
# function to split documents into smaller chunks
def text_splitter(docs:List[Document])->List[Document]:
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(minimal_docs)
    return text_chunks


In [13]:
texts_chunks=text_splitter(minimal_docs)
print("Number of text chunks:", len(texts_chunks))

Number of text chunks: 0


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

# creating embeddings for the text chunks using a Hugging Face model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [15]:
vector=embedding.embed_query("susovan paul is a good boy")
print("size of the vector:", len(vector))
print("vector:", vector)

size of the vector: 384
vector: [-0.07261820882558823, -0.022407304495573044, -0.057143066078424454, -0.035447441041469574, -0.0428701713681221, 0.08962225914001465, 0.058572012931108475, -0.0048554628156125546, -0.021207723766565323, -0.0063386959955096245, 0.049993544816970825, 0.05213024094700813, -0.0316401831805706, -0.024630611762404442, -0.02096322551369667, -0.010879883542656898, 0.08275603502988815, 0.09511151909828186, -0.02010800503194332, -0.032126329839229584, -0.08101478219032288, 0.03090761974453926, 0.08501166850328445, -0.07392802089452744, 0.020463669672608376, 0.03703225404024124, 0.014987237751483917, -0.02608376368880272, 0.04386460408568382, 0.07032429426908493, 0.06903920322656631, -0.02999131940305233, -0.043732840567827225, -0.03606877103447914, 0.05166846141219139, 0.05417894199490547, -0.021527791395783424, -0.001562964404001832, 0.04290822520852089, -0.0016281752614304423, 0.008795522153377533, -0.05368880555033684, -0.06858532875776291, -0.01780693605542183

In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [17]:
# retrieving API keys from environment variables
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [18]:
# initializing the Pinecone client
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)
pc

In [19]:
from pinecone import ServerlessSpec

# creating a Pinecone index if it doesn't exist
index_name="mood-score"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, # dimension of the sentence transformer model's output
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index=pc.Index(index_name)
index

In [20]:
from langchain_pinecone import PineconeVectorStore

# creating a Pinecone vector store from the text chunks and their embeddings
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embedding,
    index_name=index_name,
    pinecone_api_key=PINECONE_API_KEY
)

In [21]:
# Load Existing Index
from langchain_pinecone import PineconeVectorStore  

index_name="mood-score"
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embedding,
    index_name=index_name
)

In [22]:
# add extra documents to the index
doc=Document(
    page_content="This is a sample document about mood scoring.",
    metadata={"source":"sample.pdf"}
)

docsearch.add_documents([doc])

['44c93976-6ed6-4c8f-a7b4-4cfc3a9d4c6b']

In [23]:
# retrieving similar documents from the index based on a query
query="What is mood?"
similar_docs=docsearch.as_retriever(search_type="similarity",search_kwarges={"k":3})
retrieved_docs=similar_docs.invoke(query)
retrieved_docs


[Document(id='d9be2332-5a8a-4a8e-833a-2963e2f1a9c4', metadata={'source': 'data\\MHGuidebook-EBookDownload.pdf'}, page_content='5) Mood – e.g. sad; happy; irritable; angry; elevated or expan-\nsive. \n     6) Affect or facial expression – e.g. congruent or incongruent \nwith mood; flat; blunted; fluctuating. \n     7) Thought content – e.g. delusions (persistent belief that is \ninconsistent with reality), paranoia; suicidal or homicidal thoughts. \n     8) Thought processing – e.g. logical/illogical; repetitive;  dis-\njointed; tendency to go on tangent; concrete. Decelerated;  slowed; \nrapid succession  of ideas.'),
 Document(id='141126e1-a9e2-4ff1-b451-8b516ee0f68e', metadata={'source': 'data\\MHGuidebook-EBookDownload.pdf'}, page_content='III: Conditions & Issues: Mood-Related Conditions \n49 \n \nMOOD –RELATED CONDITIONS:  \nMajor Depression & Bipolar Disorder \nMajor Depression \nDepression has been generally described as a decline in mood \nthat persists for an extended period, 

In [24]:
from langchain_google_genai import GoogleGenerativeAI,ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # ✅ stable & supported
    temperature=0.7,
    google_api_key=GEMINI_API_KEY
)

c:\Users\susov\OneDrive\Desktop\MoodSphere\AI\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\susov\OneDrive\Desktop\MoodSphere\AI\venv\lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [25]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import  create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [26]:
# defining a system prompt for the AI Medical Assistant
system_prompt = """
You are an AI Student Counselor.
ROLE:
- Understand student emotions and situation
- Provide supportive, practical guidance
- Act like a real counselor (calm, caring, non-judgmental)
STUDENT CONTEXT:
You will receive:
- Previous mood score
- Current mood score
- Mood trend (increasing/decreasing/stable)
- Current situation (exam, stress, family, etc.)
Use this to:
- Compare past vs current state
- Detect improvement or decline
- Adjust your response accordingly
RISK LOGIC:
- LOW: normal stress → motivate + study help
- MEDIUM: sadness/anxiety → emotional support + coping steps
- HIGH: hopeless/self-harm → urgent support + suggest real help
RESPONSE STYLE:
1. Start with empathy
2. Understand emotion
3. Ask 1–2 open questions
4. Give simple practical suggestion
5. Add small encouragement
HIGH RISK:
- Stay calm
- Encourage contacting trusted person
- Ask: "Can I help you connect with support?"

Use this as a reference from book - {context}
OUTPUT (JSON):
{{
  "emotion": "...",
  "risk_level": "low/medium/high",
  "response": "...",
  "action": "none / book_appointment / alert"
}}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Student Info:\n{student_context}\n\nMessage:\n{input}")
])

In [27]:
# creating a question-answering chain that combines retrieved documents to answer user queries
question_answer_chain = create_stuff_documents_chain(
    chatModel,
    prompt=prompt,
)
# creating a retrieval-augmented generation (RAG) chain that retrieves similar documents and uses them to answer user queries
rag_chain=create_retrieval_chain(
    retriever=similar_docs,
    combine_docs_chain=question_answer_chain
)

In [28]:
# example student data (get from mongoDB) to test the RAG chain
student_data = {
    "previous_mood": 0.6,
    "current_mood": 0.3,
    "mood_history": [0.7, 0.6, 0.5, 0.4, 0.3],
    "trend": "decreasing",
    "situation":["exam , family issues, breakup"]
}
# formatting the student data into a context string to be used in the prompt
student_context = f"""
Previous Mood: {student_data['previous_mood']}
Current Mood: {student_data['current_mood']}
Trend: {student_data['trend']}

Situation: {", ".join(student_data['situation'])}
"""
# example input message from the student to test the RAG chain
input_message = "I'm feeling really down and hopeless lately. I don't see the point in trying anymore."
# invoking the RAG chain with the student context and input message to get a response
response=rag_chain.invoke({
    "student_context": student_context,
    "input": input_message
})

res=response["answer"]
print("Response:", res)

Response: ```json
{
  "emotion": "Hopelessness, despair, lack of motivation",
  "risk_level": "high",
  "response": "It sounds like you're going through an incredibly difficult time right now, and I hear how much pain you're in. Feeling down and hopeless, especially when you don't see the point in trying, is an incredibly heavy burden to carry, and it's completely understandable given everything you're dealing with—exams, family issues, and a breakup.\n\nI want you to know that these feelings, while intense, can be managed, and you don't have to face them alone. When you feel this way, it's a sign that you need immediate support.\n\nCould you reach out to a trusted friend, family member, or a professional counselor right now? There are people who care deeply about you and want to help you through this difficult period.\n\nIf you're feeling overwhelmed and unsure who to turn to, please know that crisis support is available 24/7. For example, in the USA, you can call or text 988 anytime.